<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://learningjournal.github.io/pub-resources/logos/scholarnest_academy.jpg" alt="ScholarNest Academy" style="width: 1400px">
</div>

####Requirement
1. Read raw data from flight_time_raw table
2. Apply transformations to time values as hour to minute interval

    1. CRS_DEP_TIME
    2. DEP_TIME
    3. WHEELS_ON
    4. CRS_ARR_TIME
    5. ARR_TIME
3. Apply transformation to TAXI_IN to make it a minute interval

In [0]:
spark.read.table("dev.spark_db.flight_time_raw").limit(3).display()

In [0]:
df = spark.read.table("dev.spark_db.flight_time_raw")
df.limit(3).display()

In [0]:
%sql

select * from dev.spark_db.flight_time_raw

####1. Read data to create a dataframe

In [0]:
flight_time_raw_df = spark.read.table("dev.spark_db.flight_time_raw")

####2. Develop logic to transform CRS_DEP_TIME to an interval

In [0]:
from pyspark.sql.functions import expr

In [0]:
step_1_df = flight_time_raw_df.withColumns({
    "CRS_DEP_TIME_HH":expr("left(lpad(CRS_DEP_TIME, 4, '0'),2)"),
    "CRS_DEP_TIME_MM":expr("right(lpad(CRS_DEP_TIME,4,'0'),2)")
})

step_2_df = step_1_df.withColumns(
    {
         "CRS_DEP_TIME_NEW":expr("cast(concat(CRS_DEP_TIME_HH, ':', CRS_DEP_TIME_MM) as interval hour to minute)")
    }
)

step_2_df.display()

In [0]:
 
step_1_df = (
    flight_time_raw_df.withColumns({
    "CRS_DEP_TIME_HH": expr("left(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
    "CRS_DEP_TIME_MM": expr("right(lpad(CRS_DEP_TIME, 4, '0'), 2)"),
    })
)

step_2_df = (
    step_1_df.withColumns({
        "CRS_DEP_TIME_NEW": expr("cast(concat(CRS_DEP_TIME_HH, ':', CRS_DEP_TIME_MM) AS INTERVAL HOUR TO MINUTE)")
    })
)

In [0]:
step_2_df.limit(2).display()

####3. Develop a reusable function

####4. Apply function to dataframe

In [0]:
def get_interval_udf(hh_mm):
    from pyspark.sql.functions import expr

    return expr(f"""
                cast(concat(left(lpad({hh_mm}, 4, '0'), 2),':', right(lpad({hh_mm}, 4, '0'), 2)) AS INTERVAL HOUR TO MINUTE)
                """)



result_df = (
    flight_time_raw_df.withColumns({
        "CRS_DEP_TIME": get_interval_udf("CRS_DEP_TIME"),
        "DEP_TIME": get_interval_udf("DEP_TIME"),
        "WHEELS_ON": get_interval_udf("WHEELS_ON"),
        "CRS_ARR_TIME": get_interval_udf("CRS_ARR_TIME"),
        "ARR_TIME": get_interval_udf("ARR_TIME"),
        "TAXI_IN": expr("cast(TAXI_IN AS INTERVAL MINUTE)")
    })
)

In [0]:
result_df.display()

####5. Save results to the table 

In [0]:
result_df.write.mode("overwrite").saveAsTable("dev.spark_db.flight_time")

In [0]:
%sql

select * from dev.spark_db.flight_time

&copy; 2021-2026 <a href="https://www.scholarnest.com/">ScholarNest</a>. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the <a href="https://www.apache.org/">Apache Software Foundation.</a><br/>
Databricks, Databricks Cloud and the Databricks logo are trademarks of the <a href="https://www.databricks.com/">Databricks Inc.</a><br/>
<a href="https://www.scholarnest.com/pages/privacy">Privacy Policy</a> | <a href="https://www.scholarnest.com/pages/terms">Terms of Use</a> | <a href="https://www.scholarnest.com/pages/contact">Contact Us</a>
